In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import root_mean_squared_error


## Data preparation

In [2]:
data = pd.read_parquet(Path.cwd().parent / "data/raw/bikes.parquet")
data.head()

,counter_id,counter_name,site_id,site_name,bike_count,date,counter_installation_date,coordinates,counter_technical_id,latitude,longitude,log_bike_count
48321,100007049-102007049,28 boulevard Diderot E-O,100007049,28 boulevard Diderot,0.0,2020-09-01 02:00:00,2013-01-18,"48.846028,2.375429",Y2H15027244,48.846028,2.375429,0.000000
48324,100007049-102007049,28 boulevard Diderot E-O,100007049,28 boulevard Diderot,1.0,2020-09-01 03:00:00,2013-01-18,"48.846028,2.375429",Y2H15027244,48.846028,2.375429,0.693147
48327,100007049-102007049,28 boulevard Diderot E-O,100007049,28 boulevard Diderot,0.0,2020-09-01 04:00:00,2013-01-18,"48.846028,2.375429",Y2H15027244,48.846028,2.375429,0.000000
48330,100007049-102007049,28 boulevard Diderot E-O,100007049,28 boulevard Diderot,4.0,2020-09-01 15:00:00,2013-01-18,"48.846028,2.375429",Y2H15027244,48.846028,2.375429,1.609438
48333,100007049-102007049,28 boulevard Diderot E-O,100007049,28 boulevard Diderot,9.0,2020-09-01 18:00:00,2013-01-18,"48.846028,2.375429",Y2H15027244,48.846028,2.375429,2.302585


In [5]:
_target_column_name = "log_bike_count"

def get_train_data(path=Path.cwd().parent / "data/raw/bikes.parquet"):
    data = pd.read_parquet(path)
    # Sort by date first, so that time based cross-validation would produce correct results
    data = data.sort_values(["date", "counter_name"])
    y_array = data[_target_column_name].values
    X_df = data.drop([_target_column_name, "bike_count"], axis=1)
    return X_df, y_array


def train_test_split_temporal(X, y, delta_threshold="30 days"):
    cutoff_date = X["date"].max() - pd.Timedelta(delta_threshold)
    mask = X["date"] <= cutoff_date
    X_train, X_valid = X.loc[mask], X.loc[~mask]
    y_train, y_valid = y[mask], y[~mask]
    return X_train, y_train, X_valid, y_valid


def _encode_dates(X):
    X = X.copy()

    X["year"] = X["date"].dt.year
    X["month"] = X["date"].dt.month
    X["day"] = X["date"].dt.day
    X["weekday"] = X["date"].dt.weekday
    X["hour"] = X["date"].dt.hour

    return X.drop(columns=["date"])


def make_preprocessor(date_cols, categorical_cols, dense_output=False):
    return ColumnTransformer(
        [
            (
                "date",
                OneHotEncoder(handle_unknown="ignore", sparse_output=not dense_output),
                date_cols,
            ),
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore", sparse_output=not dense_output),
                categorical_cols,
            ),
        ],
        sparse_threshold=0.0 if dense_output else 0.3,
    )


In [6]:
X, y = get_train_data()
X_train, y_train, X_valid, y_valid = train_test_split_temporal(X, y)

print(
    f'Train: n_samples={X_train.shape[0]},  {X_train["date"].min()} to {X_train["date"].max()}'
)
print(
    f'Valid: n_samples={X_valid.shape[0]},  {X_valid["date"].min()} to {X_valid["date"].max()}'
)

_encode_dates(X_train[["date"]]).columns.tolist()

Train: n_samples=456507,  2020-09-01 01:00:00 to 2021-08-10 23:00:00
Valid: n_samples=40320,  2021-08-11 00:00:00 to 2021-09-09 23:00:00


['year', 'month', 'day', 'weekday', 'hour']

## Lasso model

In [7]:
date_encoder = FunctionTransformer(_encode_dates)
date_cols = _encode_dates(X_train[["date"]]).columns.tolist()
categorical_cols = ["counter_name", "site_name"]

lasso_preprocessor = make_preprocessor(date_cols, categorical_cols, dense_output=False)
lasso_regressor = Lasso(alpha=0.001, max_iter=10000, random_state=0)

pipe_lasso = make_pipeline(date_encoder, lasso_preprocessor, lasso_regressor)
pipe_lasso.fit(X_train, y_train)

Pipeline(steps=[('functiontransformer',
                 FunctionTransformer(func=<function _encode_dates at 0x15aa0c680>)),
                ('columntransformer',
                 ColumnTransformer(transformers=[('date',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['year', 'month', 'day',
                                                   'weekday', 'hour']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['counter_name',
                                                   'site_name'])])),
                ('lasso', Lasso(alpha=0.001, max_iter=10000, random_state=0))])

In [ ]:
print(f"Train set, RMSE={root_mean_squared_error(y_train, pipe_lasso.predict(X_train)):.2f}")
print(f"Valid set, RMSE={root_mean_squared_error(y_valid, pipe_lasso.predict(X_valid)):.2f}")

print("Baseline mean prediction.")
print(f"Train set, RMSE={root_mean_squared_error(y_train, np.full(y_train.shape, y_train.mean())):.2f}")
print(f"Valid set, RMSE={root_mean_squared_error(y_valid, np.full(y_valid.shape, y_valid.mean())):.2f}")

In [ ]:
mask = (
    (X_valid["counter_name"] == "Totem 73 boulevard de Sébastopol S-N")
    & (X_valid["date"] > pd.to_datetime("2021/09/01"))
    & (X_valid["date"] < pd.to_datetime("2021/09/08"))
)

df_viz = X_valid.loc[mask].copy()
df_viz["bike_count"] = np.exp(y_valid[mask.values]) - 1
df_viz["bike_count (predicted)"] = np.exp(pipe_lasso.predict(X_valid[mask])) - 1

fig, ax = plt.subplots(figsize=(12, 4))
df_viz.plot(x="date", y="bike_count", ax=ax)
df_viz.plot(x="date", y="bike_count (predicted)", ax=ax, ls="--")
ax.set_title("Predictions with Lasso")
ax.set_ylabel("bike_count")

In [ ]:
fig, ax = plt.subplots()

df_viz = pd.DataFrame({"y_true": y_valid, "y_pred": pipe_lasso.predict(X_valid)}).sample(
    10000, random_state=0
)

df_viz.plot.scatter(x="y_true", y="y_pred", s=8, alpha=0.1, ax=ax)

In [ ]:
cv = TimeSeriesSplit(n_splits=6)

scores = cross_val_score(
    pipe_lasso, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error"
)
print("RMSE: ", scores)
print(f"RMSE (all folds): {-scores.mean():.3} ± {(-scores).std():.3}")

## RandomForest model

In [8]:
rf_preprocessor = make_preprocessor(date_cols, categorical_cols, dense_output=True)
rf_regressor = RandomForestRegressor(
    n_estimators=100,
    random_state=0,
    n_jobs=-1,
    max_depth=20,
    min_samples_leaf=2,
)

pipe_rf = make_pipeline(date_encoder, rf_preprocessor, rf_regressor)
pipe_rf.fit(X_train, y_train)

KeyboardInterrupt: 

In [ ]:
print(f"Train set, RMSE={root_mean_squared_error(y_train, pipe_rf.predict(X_train)):.2f}")
print(f"Valid set, RMSE={root_mean_squared_error(y_valid, pipe_rf.predict(X_valid)):.2f}")

print("Baseline mean prediction.")
print(f"Train set, RMSE={root_mean_squared_error(y_train, np.full(y_train.shape, y_train.mean())):.2f}")
print(f"Valid set, RMSE={root_mean_squared_error(y_valid, np.full(y_valid.shape, y_train.mean())):.2f}")

In [ ]:
mask = (
    (X_valid["counter_name"] == "Totem 73 boulevard de Sébastopol S-N")
    & (X_valid["date"] > pd.to_datetime("2021/09/01"))
    & (X_valid["date"] < pd.to_datetime("2021/09/08"))
)

df_viz = X_valid.loc[mask].copy()
df_viz["bike_count"] = np.exp(y_valid[mask.values]) - 1
df_viz["bike_count (predicted)"] = np.exp(pipe_rf.predict(X_valid[mask])) - 1

fig, ax = plt.subplots(figsize=(12, 4))
df_viz.plot(x="date", y="bike_count", ax=ax)
df_viz.plot(x="date", y="bike_count (predicted)", ax=ax, ls="--")
ax.set_title("Predictions with RandomForest")
ax.set_ylabel("bike_count")

In [ ]:
fig, ax = plt.subplots()

df_viz = pd.DataFrame({"y_true": y_valid, "y_pred": pipe_rf.predict(X_valid)}).sample(
    10000, random_state=0
)

df_viz.plot.scatter(x="y_true", y="y_pred", s=8, alpha=0.1, ax=ax)

In [ ]:
scores = cross_val_score(
    pipe_rf, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error"
)
print("RMSE: ", scores)
print(f"RMSE (all folds): {-scores.mean():.3} ± {(-scores).std():.3}")

## Saving models, plots and new datasets

### With pickle

In [9]:
pickle.dump(pipe_lasso, open("../models/model_lasso.pkl", "wb"))
pickle.dump(pipe_lasso, open("../models/model_random_forest.pkl", "wb"))

### Saving plots

In [ ]:
fig, ax = plt.subplots()

df_viz = pd.DataFrame({"y_true": y_valid, "y_pred": pipe_lasso.predict(X_valid)}).sample(
    10000, random_state=0
)
df_viz.plot.scatter(x="y_true", y="y_pred", s=8, alpha=0.1, ax=ax)
plt.savefig("../plots/plot_lasso_viz.png")
plt.close()

fig, ax = plt.subplots()
df_viz = pd.DataFrame({"y_true": y_valid, "y_pred": pipe_rf.predict(X_valid)}).sample(
    10000, random_state=0
)
df_viz.plot.scatter(x="y_true", y="y_pred", s=8, alpha=0.1, ax=ax)
plt.savefig("../plots/plot_random_forest_viz.png")
plt.close()